# SPS Self-Specialization — Minimal Research Prototype

This notebook demonstrates the exact concept: **State 0 (statically programmed IntegerMultiplication) → runtime request → self-replication → Ollama-guided specialization → verification → State 1 (FloatMultiplication)**.

The important behavior is that an unsupported float request does **not** simply fail. The framework uses the existing integer capability as the parent, creates a child, asks the external Ollama model to specialize it, verifies the generated capability, and activates it only after verification succeeds.

In [ ]:
# Get the latest repository directly.
%cd /content
!rm -rf self-specialization
!git clone -q https://github.com/muhammadnaumantahir/self-specialization.git
%cd /content/self-specialization
!pip -q install -r requirements.txt pytest

In [ ]:
# Deterministic tests: replication, specialization, verification and dispatch.
!PYTHONPATH=. pytest -q

## Start Ollama

Colab needs `zstd` before the current Ollama installer can unpack successfully. The previous notebook failed here and therefore reached the framework with no Ollama server.

In [ ]:
!apt-get update -qq
!apt-get install -y -qq zstd curl
!curl -fsSL https://ollama.com/install.sh | sh
!nohup ollama serve >/tmp/ollama.log 2>&1 &
!sleep 5
!ollama --version
!curl -sf http://127.0.0.1:11434/api/tags || (cat /tmp/ollama.log; exit 1)

In [ ]:
# Pull the local coding model used by the prototype.
!ollama pull qwen2.5-coder:7b

In [ ]:
# Run the actual end-to-end experiment.
import os
os.environ['OLLAMA_MODEL'] = 'qwen2.5-coder:7b'
!PYTHONPATH=. python experiments/self_specialization_demo.py

## Expected research result

The successful run should show:

```text
STATE 0 — STATIC INTEGER MULTIPLICATION
6 * 7 = 42

RUNTIME REQUEST — INTEGER
8 * 9 = 72
Ollama is not needed because State 0 already supports int × int.

RUNTIME REQUEST — FLOAT (MISSING CAPABILITY)
... replicate → Ollama specialize → verify → activate ...
Result capability: FloatMultiplication
Result state: S1
2.5 * 4.0 = 10.0

SUCCESS: State 0 reproduced and specialized into State 1.
```

The lineage should be `IntegerMultiplication → IntegerMultiplication-child → FloatMultiplication`.